# MagicTelescope classification

Your astrophysicist friend is doing an experimental project with a certain telescope. You can assume you don't know anything about astrophysics (if you do, you may be able to improve this laboratory session! Send me an email). Your friend says that there is a simulation study that classified readings from a telescope into signal and background.

Because your friend does not know about machine learning, the task has fallen to you. The experiment they are trying to replicate has the following features:

1. fLength: continuous (major axis of ellipse)
2. fWidth: continuous (minor axis of ellipse)
3. fSize: continuous (10-log of sum of content of all pixels)
4. fConc: continuous (ratio of sum of two highest pixels over fSize)
5. fConc1: continuous (ratio of highest pixel over fSize)
6. fAsym: continuous (distance from highest pixel to center, projected onto major axis)
7. fM3Long: continuous (3rd root of third moment along major axis)
8. fM3Trans: continuous (3rd root of third moment along minor axis)
9. fAlpha: continuous (angle of major axis with vector to origin)
10. fDist: continuous (distance from origin to center of ellipse)

The class to predict is `g` or `h`, meaning a gamma (signal) or a hadron (background).

> **Note:** in the provided files every column name carries a trailing colon (`fLength:`, `fAlpha:`, ...), a leftover from the original dataset format. Keep it in mind when selecting columns, e.g. `X["fAlpha:"]`.

It looks like a straightforward classification problem, with only continuous variables. Easy, right?

In [ ]:
import pyarrow  # we will be using parquet files, install pyarrow if it is not installed
import pandas as pd

In [ ]:
X = pd.read_parquet("magictelescope-data.parquet")
y = pd.read_parquet("magictelescope-labels.parquet")

Let's perform a cross validation to see if `LogisticRegression` works properly.

We have done this in class so many times that I am ashamed to ask you to write it. Here it is, same as always:

> **Tip:** `y` is loaded as a one-column DataFrame. scikit-learn accepts it but will complain with a `DataConversionWarning` ("A column-vector y was passed when a 1d array was expected"). You can get rid of it with `y = y.iloc[:, 0]` (or `y.squeeze()`) whenever you want.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

clf_linear = LogisticRegression()
cross_validate(clf_linear, X, y, cv=3)

... wait, that is new! Why is this failing?

Mmm, it seems that your friend's telescope data has some missing values.

If you read the error message, scikit-learn is trying to help you by pointing you to the [`impute` module documentation](https://scikit-learn.org/stable/modules/impute.html). Create a `SimpleImputer` transformer to address the issue.

Show that the `SimpleImputer` is working. A simple `.fit_transform()` call is enough, followed by something like `np.isnan(...).sum()` on the result (note that `.fit_transform()` returns a NumPy array; wrap it in a `pd.DataFrame` if you prefer to use `.head()` or `.isna().sum()`).

In [ ]:
...

Of course, the next step should be to create a pipeline and try again. But... your sixth sense is tingling. Are all the values fixed now?

We forgot to explore the DataFrame! Let's address that now. Call `.describe()` on the DataFrame. Look closely at the statistics and distribution... isn't the column `fAlpha:` suspicious? Look at its values, poke around a little. Maybe there are some values that should not be like that. Use another `SimpleImputer` to fix them (hint: the `missing_values` parameter can be useful!).

In [ ]:
...

Create a pipeline with the required `SimpleImputer` instances and the `LogisticRegression`. Run `cross_validate` and see if we are getting somewhere now.

In [ ]:
...

You will probably get a `ConvergenceWarning` (if you have not, use `LogisticRegression(max_iter=10)` to see the warning).

Answer the following questions _very briefly_. Write your answers in the markdown cell right below. You can run some tests yourself in the code cells after it (your playground) to make sure that your answers are correct:
- Can `max_iter` affect the quality of a model?
- Will a converged model always perform better (have a higher score) than a non-converged one?
- Given two converged models, one with a `StandardScaler` transformer and one without, would their scores differ significantly?
- Can the presence of a `StandardScaler` transformer affect the number of iterations?

...

In [ ]:
...

At this point we could do some more model selection, hyperparameter tuning (e.g. if using an ensemble), and so on. At the end, we would finally `.fit()` the model on all our training data to obtain the final model (this includes the transformers and everything, of course).

Now is the time to test the performance of the model on a separate testing set:

In [ ]:
X_test = pd.read_parquet("magictelescope-testdata.parquet")
y_test = pd.read_parquet("magictelescope-testlabels.parquet")

Fit your pipeline on the whole training set (`X`, `y`), keep it in a variable called `model`, and score it with the test set: `model.score(X_test, y_test)`.

In [ ]:
...

Wait, that score is awful!

What have you done?

Ok, joking aside: the original performance (given by cross validation) was close to 80%, and now this test dataset gives a score below 50% (worse than always predicting the most frequent class!). Surely there is something wrong with the dataset.

Explore the `X_test` DataFrame and compare how the data is distributed in both datasets. If you had a lot of time, now would be the moment to do a side-by-side violin plot and look for meaningful deviations between them. But for this simple example, it should be enough to look firmly at the `.describe()` output of both. Which column is _very_ different between `X_test` and the training data? Why?

Fix it and score the model again. Explain which column is different and why in the markdown cell after the code cells.

In [ ]:
...

...

## Extra content

_Because your professor is awful at estimating the length of laboratory sessions._

Unless you reach this point very quickly, consider these exercises optional.

We were able to fix the mismatch in the `fAlpha:` feature because its semantics were clear to us.

Imagine that we did not know the conversion factor. A plausible workaround is to do the standardization outside the pipeline, separately for each dataset (it can make sense if there is a lot of data, but it is not an ideal scenario).

Without using pipelines: apply the imputers and a `StandardScaler` to the training DataFrame and train the model on the result. Then standardize the (original, unfixed) test dataset with a **different** `StandardScaler`, fitted on the test data itself, and score the model on it.

The score should be similar to the one you got after fixing the units. This "trick" only works when both datasets cover almost all of the data space; this is desirable and somewhat expected for the training dataset, but there is no such guarantee at the inference stage.

In [ ]:
...

Go back to the point where we introduced the basic `SimpleImputer`. It might be enough in some scenarios (an alternative is to drop rows entirely). There are other imputer transformers, such as the `KNNImputer`. This nearest-neighbors-based imputer fills each missing value according to the nearest neighbors (computed using the other features). It is **much** slower (it must search the neighbors of every row with a `nan`), but it can bring benefits (often depending on whether there is a hidden correlation across the features, which may or may not be the case). Replace the `SimpleImputer` with a `KNNImputer` and compare the cross validation scores.

In [ ]:
...